In [3]:
import os
import time
import requests
from pathlib import Path
from getpass import getpass

In [4]:
os.environ["LM_USER"] = input("Account name: ")
os.environ["LM_PASS"] = getpass("Password: ")

Account name:  uran0001
Password:  ········


In [2]:
BASE = "https://api.lantmateriet.se/stac-vektor/v1"
COLLECTION = "byggnader"
OUT = Path("byggnader_sverige")
OUT.mkdir(exist_ok=True)

In [6]:
auth = (os.environ["LM_USER"], os.environ["LM_PASS"])

session = requests.Session()
session.auth = auth

In [7]:
def iter_items():
    """Walk through every item in the collection, following 'next' links."""
    url = f"{BASE}/collections/{COLLECTION}/items"
    params = {"limit": 100}
    while url:
        r = session.get(url, params=params, timeout=60)
        r.raise_for_status()
        page = r.json()
        yield from page["features"]
        url = next((l["href"] for l in page.get("links", [])
                    if l.get("rel") == "next"), None)
        params = None  # the next link already contains the query

In [8]:
def download(url, dest):
    if dest.exists():
        print(f"  skip (already have) {dest.name}")
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, 4):
        try:
            with session.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            tmp.rename(dest)
            return
        except requests.RequestException as e:
            print(f"  attempt {attempt} failed: {e}")
            time.sleep(5 * attempt)
    print(f"  GAVE UP on {url}")

In [9]:
items = list(iter_items())
print(f"Found {len(items)} items")

for n, item in enumerate(items, 1):
    print(f"[{n}/{len(items)}] {item['id']}")
    for name, asset in item["assets"].items():
        href = asset["href"]
        filename = Path(href.split("?")[0]).name or f"{item['id']}_{name}"
        download(href, OUT / filename)

Found 290 items
[1/290] 2581
[2/290] 2584
[3/290] 2580
[4/290] 2583
[5/290] 2582
[6/290] 2523
[7/290] 2514
[8/290] 2560
[9/290] 2521
[10/290] 2482
[11/290] 2518
[12/290] 2480
[13/290] 2513
[14/290] 2510
[15/290] 2506
[16/290] 2505
[17/290] 2481
[18/290] 2462
[19/290] 2463
[20/290] 2460
[21/290] 2425
[22/290] 2421
[23/290] 2422
[24/290] 2409
[25/290] 2418
[26/290] 2380
[27/290] 2401
[28/290] 2417
[29/290] 2361
[30/290] 2404
[31/290] 2403
[32/290] 2284
[33/290] 2326
[34/290] 2313
[35/290] 2321
[36/290] 2309
[37/290] 2305
[38/290] 2283
[39/290] 2282
[40/290] 2303
[41/290] 2281
[42/290] 2184
[43/290] 2180
[44/290] 2183
[45/290] 2262
[46/290] 2280
[47/290] 2181
[48/290] 2161
[49/290] 2260
[50/290] 2182
[51/290] 2132
[52/290] 2121
[53/290] 2104
[54/290] 2085
[55/290] 2101
[56/290] 2081
[57/290] 2084
[58/290] 2082
[59/290] 2083
[60/290] 2080
[61/290] 2062
[62/290] 2039
[63/290] 2061
[64/290] 2034
[65/290] 2029
[66/290] 2031
[67/290] 2023
[68/290] 2026
[69/290] 1980
[70/290] 1981
[71/290] 2021